In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS learning.lab COMMENT "scratch area for expriments";
CREATE VOLUME IF NOT EXISTS learning.lab.files COMMENT "path-based Delta tables for learning internals";

In [0]:
PATH = "/Volumes/learning/lab/files/delta_demo"
df = spark.createDataFrame(
    [(1, "Anna", "Germany"), (2, "Lucas", "France"), (3, "Sara", "Spain")],
    "id INT, name STRING, country STRING"
)

df.write.format("delta").mode("overwrite").save(f"{PATH}")

display(dbutils.fs.ls(PATH))
display(dbutils.fs.ls(f"{PATH}/_delta_log"))

In [0]:
print(dbutils.fs.head(f"{PATH}/_delta_log/00000000000000000000.json"))

In [0]:
PATH = "/Volumes/learning/lab/files/delta_demo"
print(dbutils.fs.head(f"{PATH}/_delta_log/00000000000000000001.json"))

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/learning/lab/files/delta_demo`

In [0]:
print("Delta  :", spark.read.format("delta").load(PATH).count())   # 3: follows the log


In [0]:
print("Delta  :", spark.read.format("delta").load(PATH).count())   # 3: follows the log


In [0]:
new_rows = spark.createDataFrame(
    [(4, "Pedro", "Spain"), (5, "Sofia", "France")],
    "id INT, name STRING, country STRING"
)
new_rows.write.format("delta").mode("append").save(f"{PATH}")
print(dbutils.fs.head(f"{PATH}/_delta_log/00000000000000000002.json"))

In [0]:
print("rows:", spark.read.format("delta").load(PATH).count())
display(dbutils.fs.ls(PATH))

In [0]:
PATH = "/Volumes/learning/lab/files/delta_demo"

display(spark.sql(f"DESCRIBE DETAIL delta.`{PATH}`").select("numFiles", "tableFeatures", "properties"))

In [0]:
spark.sql(f"DELETE FROM delta .`{PATH}` WHERE id = 2")
print(dbutils.fs.head(f"{PATH}/_delta_log/00000000000000000003.json"))
display(dbutils.fs.ls(PATH))

In [0]:
display(dbutils.fs.ls(f"{PATH}/_delta_log"))

In [0]:
print(dbutils.fs.head(f"{PATH}/_delta_log/00000000000000000004.json"))


In [0]:
print("rows after OPTIMIZE:", spark.read.format("delta").load(PATH).count())
display(dbutils.fs.ls(PATH))

In [0]:
display(dbutils.fs.ls(PATH))

In [0]:
spark.sql(f"UPDATE delta .`{PATH}` SET country = 'Portugal' WHERE id = 4")
print(dbutils.fs.head(f"{PATH}/_delta_log/00000000000000000005.json"))
display(dbutils.fs.ls(PATH))
print("rows after UPDATE:", spark.read.format("delta").load(PATH).count)

In [0]:
%sql
SELECT "v2" AS version, COUNT(*) AS n_rows FROM delta.`/Volumes/learning/lab/files/delta_demo` VERSION AS OF 2
UNION ALL
SELECT "now", COUNT(*) FROM delta.`/Volumes/learning/lab/files/delta_demo`;